<a href="https://colab.research.google.com/github/hibameo/langchain/blob/main/langchain_RAG_project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install langchain
!pip install tqdm
!pip install langchain-pinecone



   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 18.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 427.3/427.3 kB 23.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.5/87.5 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.3/50.3 kB 3.0 MB/s eta 0:00:00
  Attempting uninstall: aiohttp
    Found existing installation: aiohttp 3.11.11
    Uninstalling aiohttp-3.11.11:
      Successfully uninstalled aiohttp-3.11.11


In [ ]:
from google.colab import userdata
import os
PINECONE_API_KEY = userdata.get('PINECONE_API_KEY')
os.environ['PINECONE_API_KEY'] = PINECONE_API_KEY

GOOGLE_API_KEY = userdata.get('GOOGLE_API_KEY_4')
os.environ['GOOGLE_API_KEY_3'] = GOOGLE_API_KEY
PINECONE_ENVIRONMENT = 'us-east-1'

SecretNotFoundError: Secret PINECONE_API_KEY does not exist.

In [ ]:
from pinecone import Pinecone, ServerlessSpec


pc = Pinecone(
    api_key=PINECONE_API_KEY
)

# Check if the index exists; if not, create it
index_name = "anewindex"
if index_name not in pc.list_indexes().names():
    pc.create_index(
        name=index_name,
        dimension=768,
        metric="cosine",  # Choose the metric: cosine, euclidean, or dotproduct
        spec=ServerlessSpec(
            cloud="aws",
            region=PINECONE_ENVIRONMENT  # Use your environment's region
        )
    )

# # Connect to the index
index = pc.Index(name=index_name)
print(f"Successfully connected to index: {index_name}")


Successfully connected to index: anewindex


In [ ]:
!pip install langchain-google-genai

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.3/41.3 kB 1.7 MB/s eta 0:00:00


In [ ]:
from langchain_google_genai.embeddings import GoogleGenerativeAIEmbeddings

# os.environ["PINECONE_API_KEY"] = PINECONE_API_KEY # Assuming PINECONE_API_KEY is already defined


embeddings = GoogleGenerativeAIEmbeddings(
    model="models/embedding-001",  # Specify the desired embedding model
    api_key=GOOGLE_API_KEY
)

In [ ]:
!pip install langchain-community

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 32.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 47.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 411.6/411.6 kB 29.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 kB 3.9 MB/s eta 0:00:00
  Attempting uninstall: langchain-core
    Found existing installation: langchain-core 0.3.25
    Uninstalling langchain-core-0.3.25:
      Successfully uninstalled langchain-core-0.3.25
  Attempting uninstall: langchain
    Found existing installation: langchain 0.3.12
    Uninstalling langchain-0.3.12:
      Successfully uninstalled langchain-0.3.12


In [ ]:
from langchain.document_loaders import TextLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter

# Load documents
loader = TextLoader("/content/document.txt")  # Replace with your file
documents = loader.load()

# Split documents into chunks
text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
docs = text_splitter.split_documents(documents)

RuntimeError: Error loading /content/document.txt

In [ ]:
!pip install langchain-google-genai
!pip install langchain-community
!pip install tqdm
from langchain_google_genai.embeddings import GoogleGenerativeAIEmbeddings
from langchain.document_loaders import TextLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from tqdm import tqdm
from google.colab import userdata
import os
import time

# Configure Google API Key
GOOGLE_API_KEY = userdata.get('GOOGLE_API_KEY_4')
os.environ['GOOGLE_API_KEY'] = GOOGLE_API_KEY # Ensure it's GOOGLE_API_KEY, not GOOGLE_API_KEY_3

# Initialize embeddings
embeddings = GoogleGenerativeAIEmbeddings(
    model="models/embedding-001",
    api_key=GOOGLE_API_KEY
)

# Load and split documents
loader = TextLoader("/content/document.txt")
documents = loader.load()
text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
docs = text_splitter.split_documents(documents)

# Embedding and upserting with retry logic
for doc in tqdm(docs):
    retries = 3  # Number of retries
    delay = 5  # Delay between retries in seconds

    for i in range(retries):
        try:
            vector = embeddings.embed_query(doc.page_content)
            index.upsert([{
                "id": doc.metadata["source"],
                "values": vector,
                "metadata": {
                    "text": doc.page_content,
                    "source": doc.metadata["source"]
                }
            }])
            break  # Exit retry loop if successful
        except GoogleGenerativeAIError as e:
            if "Timeout" in str(e) and i < retries - 1:
                print(f"Timeout error, retrying in {delay} seconds...")
                time.sleep(delay)
            else:
                print(f"Error embedding document: {e}")
                # Consider logging the error for further investigation
                break # Exit retry loop if other errors occur

100%|██████████| 29/29 [00:06<00:00,  4.38it/s]


In [ ]:
from tqdm import tqdm

# Create embeddings and upload to Pinecone
for doc in tqdm(docs):
    vector = embeddings.embed_query(doc.page_content)

    # Modify the upsert to use a dictionary for metadata
    index.upsert([{
        "id": doc.metadata["source"],  # Use "id" instead of the first tuple element
        "values": vector,  # Use "values" for the vector
        "metadata": {
            "text": doc.page_content,  # Add the text as part of metadata
            "source": doc.metadata["source"]  # Include the source
        }
    }])

100%|██████████| 29/29 [00:07<00:00,  4.08it/s]


In [ ]:
from langchain.vectorstores import Pinecone

# Use from_existing_index to load from an existing index
retriever = Pinecone.from_existing_index(index_name=index_name, embedding=embeddings, text_key="text")

In [ ]:
from langchain_google_genai import ChatGoogleGenerativeAI

gemini_model = ChatGoogleGenerativeAI(api_key=GOOGLE_API_KEY,model="gemini-1.5-flash", temperature=0.7)

In [ ]:
from langchain_pinecone import PineconeVectorStore
from langchain.chains import RetrievalQA

# Create a vector store using the Pinecone index
vectorstore = PineconeVectorStore(
    index_name=index_name,
    embedding=embeddings
)

# Create the retriever
retriever = vectorstore.as_retriever(search_kwargs={"k": 2})  # Retrieve top 2 most similar documents

# Create the QA chain
qa_chain = RetrievalQA.from_chain_type(
    llm=gemini_model,
    chain_type="stuff",
    retriever=retriever,  # Pass the retriever here
    return_source_documents=True  # Optional: to get the source documents used in the response
)

In [ ]:
print("Welcome To CODE WITH HIBA")
print('='*40)

query = "What is the Future of Agentic AI?"
print('Human Message:',query)
response = qa_chain.invoke(query)

# Print the answer
print("Agent Message:", response['result'])

# Print the source documents
# print("\n According To Given Information:")
for doc in response['source_documents']:
    print(f"- {doc.page_content[:200]}...")  # Print first 200 characters of each source document

Welcome To CODE WITH HIBA
Human Message: What is the Future of Agentic AI?


Agent Message: Based on the provided text, the future of Agentic AI involves the development of smarter, more adaptive systems capable of autonomously generating solutions, creating content, and interacting with the world in more natural, human-like ways.

- The progress in Generative AI and Agentic AI will lead to smarter, more adaptive systems that can autonomously generate solutions, create content, and interact with the world in more natural, human-li...


In [ ]:
print("Welcome To CODE WITH HIBA")
print('=' * 40)

query = "What is the Future of AI Agents?"
print('Human Message:', query)

# Invoke the QA chain
try:
    response = qa_chain.invoke(query)
    # Print the agent's response
    print("Agent Message:", response.get('result', 'No result found.'))
except Exception as e:
    print("Error:", e)


Welcome To CODE WITH HIBA
Human Message: What is the Future of AI Agents?
Agent Message: Based on the provided text, the future of AI agents involves smarter, more adaptive systems capable of autonomously generating solutions, creating content, and interacting with the world in more natural, human-like ways.



In [ ]:
matches = response.get('matches', [])
for i, match in enumerate(matches):
    metadata = match.get('metadata', None)
    if metadata:
        print(f"Match {i} Metadata: {metadata}")
    else:
        print(f"Match {i} has no metadata.")


In [ ]:
print(response)


{'query': 'What is the Future of Agentic AI?', 'result': 'Based on the provided text, the future of Agentic AI involves the development of smarter, more adaptive systems capable of autonomously generating solutions, creating content, and interacting with the world in more natural, human-like ways.\n', 'source_documents': [Document(id='/content/document.txt', metadata={'source': '/content/document.txt'}, page_content='The progress in Generative AI and Agentic AI will lead to smarter, more adaptive systems that can autonomously generate solutions, create content, and interact with the world in more natural, human-like ways.'), Document(id='/content/rag.ipynb', metadata={'source': '/content/rag.ipynb'}, page_content='"\\u001bPineconeApiAttributeError\\u001b: ScoredVector has no attribute \'metadata\' at [\'[\'received_data\', \'matches\', 0]\'][\'metadata\']"\n          ]\n        }\n      ]\n    }\n  ]\n}')]}


In [ ]:
# Assuming 'embeddings' is your GoogleGenerativeAIEmbeddings instance
vector = embeddings.embed_query("Placeholder text for embedding") # Replace "Placeholder text for embedding" with relevant text

index.upsert([
    {"id": "1", "values": vector, "metadata": {"key": "value"}}
])

{'upserted_count': 1}